In [ ]:
import pandas as pd
import plotly.express as px
from transformers import AutoTokenizer
import numpy as np
from pathlib import Path
import pickle
from datasets import load_dataset
import sys

# Add parent directory to path for imports
sys.path.append(str(Path().resolve().parent))

from utils import hidden

In [2]:
ds = load_dataset(
    "NicolaiSivesind/human-vs-machine", "wiki_labeled", token=hidden
).with_format("torch")

Using the latest cached version of the dataset since NicolaiSivesind/human-vs-machine couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'wiki_labeled' at /home/microslaw/.cache/huggingface/datasets/NicolaiSivesind___human-vs-machine/wiki_labeled/0.0.0/c766e6943b44bf27432a595c0a226e5c50c5405d (last modified on Wed Dec 17 02:44:17 2025).


In [3]:
df_dataset = pd.concat(
    [
        pd.DataFrame({"text": ds["test"]["text"], "generated": ds["test"]["label"]}),
        pd.DataFrame({"text": ds["train"]["text"], "generated": ds["train"]["label"]}),
        pd.DataFrame({"text": ds["validation"]["text"], "generated": ds["validation"]["label"]}),
    ]
)

In [4]:
total_len = len(df_dataset)
total_ai = (df_dataset["generated"] == 1).sum()
total_human = (df_dataset["generated"] == 0).sum()

print(
    f"The dataset consists of {total_len} samples, of which {total_ai} are ai generated and {total_human} are human written."
)

The dataset consists of 300000 samples, of which 150000 are ai generated and 150000 are human written.


In [5]:
df_dataset["text_len"] = df_dataset["text"].apply(lambda x: len(x))
df_dataset["word_count"] = df_dataset["text"].str.count(" ") + 1
median_char = df_dataset["text_len"].median().item()
median_word = df_dataset["word_count"].median().item()
print(f"Median word count equals {median_word} while median character count {median_char}")

Median word count equals 165.0 while median character count 1011.0


In [6]:
fig = px.histogram(
    df_dataset["text_len"],
    labels={
        "value": "Length of a specific entry",
        "count": "Count",
    },
)

fig

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def tokenize_function(examples):
    return tokenizer(
        examples,
        padding="longest",
    )


df_subset = df_dataset.sample(10_000)
tokenized = tokenize_function(list(df_subset["text"].to_numpy()))
tokenized = np.array(tokenized["attention_mask"])
df_subset["token_size"] = tokenized.sum(axis=1)
df_subset["truncated"] = df_subset["token_size"] > 512

fig = px.scatter(
    df_subset,
    "text_len",
    "token_size",
    color="truncated",
    labels={
        "token_size": "Number of generated tokens",
        "text_len": "Length of the text",
        "truncated": "Is output truncated",
    },
)
fig

Token indices sequence length is longer than the specified maximum sequence length for this model (1201 > 512). Running this sequence through the model will result in indexing errors


In [8]:
truncate_loss = df_subset["truncated"].mean()

print(f"We loose about {truncate_loss * 100}%")

We loose about 0.24%


In [9]:
model_dir = "models"


model_files = [file for file in Path.iterdir(Path(f"../{model_dir}")) if not Path.is_dir(file)]

model_dict = {}
for file in model_files:
    with open(file, "rb") as f:
        model_dict[file] = pickle.load(f)
model_dict

def get_feature_importances(model):
    if hasattr(model, "coef_"):
        return model.coef_.ravel()
    elif hasattr(model, "feature_importances_"):
        return model.feature_importances_


df_features = (
    pd.DataFrame(
        {
            model_path.stem: get_feature_importances(model)
            for model_path, model in model_dict.items()
        }
    )
    .reset_index(drop=False)
    .rename(columns={"index": "feature_num"})
)
df_features.head()

,feature_num,linear_svc,random_forest,logistic_regression,sgd,decision_tree
0,0,0.015771,0.000565,0.009475,-0.925300,0.000000
1,1,-0.054973,0.000628,0.094738,1.086389,0.000000
2,2,-0.005102,0.000919,-0.058403,-0.586613,0.000416
3,3,-0.464855,0.000716,-1.494119,-3.328143,0.000000
4,4,0.142653,0.000562,0.512973,1.979297,0.000000


## Model Correlation Analysis

**Two complementary analyses:**

### 1. Feature-Level Correlation
- Correlates model **weights/importances** across 768 BERT dimensions
- **Question:** "Do models rely on the same embedding dimensions?"

In [19]:
df_features = df_features[
    ["logistic_regression", "sgd", "linear_svc", "random_forest", "decision_tree"]
]
df_corr = df_features.corr(method="spearman")
fig = px.imshow(
    df_corr,
    color_continuous_midpoint=0,
    color_continuous_scale="RdBu",
)
fig.data[0].text = df_corr.round(3).to_numpy()
fig.data[0].texttemplate = "%{text}"
fig.update_traces(textfont_size=18)
fig

### 2. Prediction-Level Correlation
- Correlates actual **predictions** across 10K test samples
- **Question:** "Do models agree on which specific texts are AI/Human?"
- Split by true label: Agreement on AI samples vs Human samples

In [18]:
# Option 2: Prediction-level correlation (sample-by-sample agreement)
from datasets import load_from_disk

dataset = load_from_disk("../data/processed/AI_Human/encoded/bert_bert/test")
X_test = np.array(dataset["embeddings"])
y_test = np.array(dataset["label"])


y_data = {"ground_truth": y_test}
for model_path, model in model_dict.items():
    model_name = model_path.stem
    y_data[model_name] = model.predict(X_test)
    accuracy = (y_data[model_name] == y_test).mean()

pred_df = pd.DataFrame(y_data)[
    ["logistic_regression", "linear_svc", "sgd", "random_forest", "decision_tree"]
]

np_pred = pred_df.to_numpy(dtype=np.bool)

# first dot product computes how many predictions "1" were aligned, the second one how many "0"
agreement_matrix = pred_df.T.dot(pred_df) + (1 - pred_df).T.dot(1 - pred_df)

agreement_df = pd.DataFrame(
    agreement_matrix / len(pred_df),
    index=pred_df.columns,
    columns=pred_df.columns,
)

fig = px.imshow(
    agreement_df,
)
fig.data[0].text = agreement_df.round(3).to_numpy()
fig.data[0].texttemplate = "%{text}"
fig.update_traces(textfont_size=18)
fig